# Le réchauffement climatique


#### Elisa Leblond 21302500

## Introduction :

Dans ce projet, nous allons essayer répondre à la question : Est-ce que des mesures de température mesurées localement au cours du dernier siècle mettent-elles en évidence un réchauffement significatif ? On va travailler pour répondre à cette question sur les relevés de température enregistrés quotidiennement depuis juin 1920 à [la station météorologique de Montélimar](https://donneespubliques.meteofrance.fr/metadonnees_publiques/fiches/fiche_26198001.pdf). Elles peuvent être téléchargées librement depuis le [site de l'ECAD](https://www.ecad.eu/) (European Climate Assessment & Dataset).

Dans le fichier *montelimar_temperature.dat* un fichier de données préparées avec :
- colonne 1 : date en MJD (*modified Julian Day*)
- colonne 2 : température en degrés Celsius


La première problématique est de faire apparaître un effet faible et lent (on voit dans la figure ci-dessus que le réchauffement est de l'ordre de 1 °C sur les 30 dernières dernières années) à l'échelle des données (amplitudes de fluctuations quotidiennes ou saisonnières dix fois supérieures typiquement).

La seconde problématique est de montrer que l'effet observé est significatif, c'est-à-dire que cette augmentation des températures ne correspond pas à une fluctuation statistique. Pour cela on supposera que l'erreur sur les mesures de température est de l'ordre de 5 °C. C'est l'ordre de grandeur des fluctuations quotidiennes qui ne seront pas prises en compte dans un modèle qui décrit des variations saisonnières.

Les variations saisonnières de la température peuvent être modélisée par une sinusoïde de période une année. Il faut utiliser un modèle du type sinusoïdal :

$$ T(t) = A \sin{(\omega t + \phi)} + B $$

où les paramètres $A$ (amplitude), $\phi$ (phase) et $B$ (température moyenne) doivent être ajustés aux données, alors que $\omega = 2\pi/1\text{ an}$. Cependant, un tel modèle ajusté sur l'ensemble des données ne donnera aucune augmentation moyenne de la température. On pourra par exemple essayer d'appliquer un ajustement sinusoïdal pour chaque décennie, et voir si le paramètre $B$ (température moyenne) augmente. On peut aussi affiner ce modèle en se disant que l'accroissement lent de la température est linéaire. On pourra alors utiliser un modèle de type :

$$ T(t) = A \sin{(\omega t + \phi)} + B + C t $$

où $C$ est un nouveau paramètre à ajuster, qui correspond à l'accroissement linéaire lent de la température.

On gardera aussi en tête que l'ajustement par la fonction `curve_fit` permet de calculer assez facilement l'incertitude sur les paramètres de l'ajustement. Cela permettra de statuer sur le caractère significatif du réchauffement. On veillera alors à discuter les erreurs estimées sur les paramètres ajustés.

In [1]:
#Importation des modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from datetime import timedelta

In [4]:
# Charger les données
fichier = pd.read_csv("montelimar_temperature.dat", delim_whitespace=True, names=["MJD", "Température"]) 
print(fichier)

# generation temps
fichier["Date"] = pd.to_datetime('1858-11-17') + pd.to_timedelta(fichier["MJD"], unit='D')
fichier["Année"] = fichier["Date"].dt.year

# Tracé des températures brutes
plt.figure(figsize=(12, 5))
plt.plot(fichier["Année"], fichier["Température"], '-', alpha=0.3, label='Température quotidienne', color='m')
plt.xlabel("Année")
plt.ylabel("Température (deg)")
plt.title("Températures quotidiennes à Montélimar depuis 1920")
plt.legend()
plt.show()

# creation de la moyenne sur 365 jours
fichier["Température_Moyenne"] = fichier["Température"].rolling(window=365, min_periods=1).mean()

# Tracé de la tendance

plt.figure(figsize=(12, 5))
plt.plot(fichier["Année"], fichier["Température"], '.', markersize=1, alpha=0.3, label='Température quotidienne')
plt.plot(fichier["Année"], fichier["Température_Moyenne"], 'r-', linewidth=2, label='Moyenne mobile annuelle')
plt.xlabel("Année")
plt.ylabel("Température (deg)")
plt.title("Tendance des températures à Montélimar (1920 - aujourd'hui)")
plt.legend()
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'montelimar_temperature.dat'

### **MODELISATION DES VAIRATIONS SAISONNIERES A L'AIDE D'UN MODELE SINUSOIDAL**

La première résolution de cette problématique est faites à partir d'un modèle sinusoidal de prériode un an, en prenant en compte une erreur de 5 degrés sur les températures données. On verra cependant par la suite que cela reste un modèle peu précis. 

In [5]:
# Définition du modèle sinusoïdal
def mod_sin(t, A, omega, phi, B):
    return A * np.sin(omega * t + phi) + B

# omega periode de 1 an
omega = 2 * np.pi / 1  

# Normalisation des années (centrer autour de 1920 par exemple)
mean_year = fichier["Année"].mean()
fichier["Année_norm"] = fichier["Année"] - mean_year  # Années normalisées autour de la moyenne


# Ajustement du modèle avec les valeurs initiales adaptées
paramètres, covariance = curve_fit(mod_sin, fichier["Année_norm"], fichier["Température"], p0=[10, omega, 0, 15])

# Extraction des paramètres ajustés et de leur incertitude
A, omega, phi, B = paramètres
erreurs = np.sqrt(np.diag(covariance))

# Affichage paramètres (et incertitudes)
print(f"Amplitude A = {A:.2f} ± {erreurs[0]:.2f}")
print(f"Phase phi = {phi:.2f} ± {erreurs[2]:.2f}")
print(f"Température moyenne B = {B:.2f} ± {erreurs[3]:.2f}")

# Graphique mesures pour modèle sin avec les barres d'erreur
plt.figure(figsize=(12, 5))
plt.errorbar(fichier["Année"], fichier["Température"], yerr=5, fmt='.', markersize=1, alpha=0.3, label='Température quotidienne', elinewidth=1, capsize=3)
plt.plot(fichier["Année"], mod_sin(fichier["Année"], *paramètres), 'r-', label='Ajustement sinusoïdal')
plt.xlabel("Année")
plt.ylabel("Température (°C)")
plt.title("Ajustement sinusoïdal des températures à Montélimar avec erreurs")
plt.legend()
plt.show()

NameError: name 'fichier' is not defined

### **MODELISATION DES VAIRATIONS SAISONNIERES A L'AIDE D'UN MODELE LINEAIRE**

La seconde résolution de cette problématique est faites à partir d'un modèle sinusoidal de prériode un an, en prenant également en compte l'erreur de 5 degrés sur les températures données. Ici le modèle est assez bien ajusté pour être plus précis que le précédent, à l'aide du paramètre C.

In [6]:
# Définition du modèle avec un terme linéaire supplémentaire
def mod_lin(t, A, omega, phi, B, C):
    return A * np.sin(omega * t + phi) + B + C * t

# Ajustement du modèle
paramètres, covariance = curve_fit(mod_lin, fichier["Année"], fichier["Température"], p0=[10, omega, 0, 15, 0.01])

# Extraction des paramètres ajustés et de leur incertitude
A, omega, phi, B, C = paramètres
erreurs = np.sqrt(np.diag(covariance))

# Affichage parmètres et erreurs
print(f"Amplitude A = {A:.2f} ± {erreurs[0]:.2f}")
print(f"Phase phi = {phi:.2f} ± {erreurs[2]:.2f}")
print(f"Température moyenne B = {B:.2f} ± {erreurs[3]:.2f}")
print(f"Croissance linéaire C = {C:.4f} ± {erreurs[4]:.4f}")

# Tracé des mesures avec le modèle ajusté (ajout de barres d'erreur de 5°C)
plt.errorbar(fichier["Année"], fichier["Température"], yerr=5, fmt='.', markersize=1, alpha=0.3, label='Température quotidienne', elinewidth=1, capsize=3)
plt.plot(fichier["Année"], mod_lin(fichier["Année"], *paramètres), 'r-', label='Ajustement sinusoïdal et linéaire')
plt.xlabel("Année")
plt.ylabel("Température (°C)")
plt.title("Ajustement sinusoïdal avec tendance linéaire des températures à Montélimar")
plt.legend()
plt.show()

# Vérification paramètre C
# si l'incertitude sur C est faible par rapport à la valeur estimée, alors le réchauffement est significatif
if abs(C) > 2 * erreurs[4]: # écarts-types
    print(f"Le réchauffement linéaire (C) est significatif : {C:.4f} ± {erreurs[4]:.4f}")
else:
    print(f"Le réchauffement linéaire (C) n'est pas significatif : {C:.4f} ± {erreurs[4]:.4f}")

NameError: name 'fichier' is not defined